# Data Lake

In [1]:
import pandas as pd
import numpy as np
from minio import Minio
import os

## Generate data

In [2]:
data_dict = {
    'name': [''.join(np.random.choice(list('abcdefghijklmnopqrstuvwxyz'), size=8)) for _ in range(100)],
    'price': np.random.random(100).tolist(),
    'quantity': np.random.randint(1, 10_000, size=100).tolist(),
}

df = pd.DataFrame(data_dict)
df.head()

,name,price,quantity
0,tbxpznnw,0.551947,8147
1,piejchhh,0.154326,2705
2,sbnmgkjx,0.460766,1818
3,ivumatqy,0.162864,1522
4,xmrmeiua,0.745442,3936


In [3]:
split_idx = np.array_split(df.index, 3)
df_part_1, df_part_2, df_part_3 = [df.loc[idx].reset_index(drop=True) for idx in split_idx]
df_part_1, df_part_2, df_part_3

(        name     price  quantity
 0   tbxpznnw  0.551947      8147
 1   piejchhh  0.154326      2705
 2   sbnmgkjx  0.460766      1818
 3   ivumatqy  0.162864      1522
 4   xmrmeiua  0.745442      3936
 5   wqqzgyav  0.058225      1840
 6   ztgeldgb  0.603500      6613
 7   gwsiaskn  0.729337       728
 8   cjidpcnh  0.701489      2334
 9   hzyadkui  0.926079       643
 10  uqggrkuc  0.362639      5479
 11  qdrxvnbc  0.072988      7078
 12  mysgiftt  0.151491      9181
 13  ajratgqk  0.856017      2438
 14  zvoqwqwb  0.380585      8045
 15  ltkqfzdp  0.880036      8428
 16  degqadal  0.435329      7465
 17  xabpqkmr  0.192446       548
 18  wkxljgkq  0.307241      2726
 19  vioopban  0.821919      5246
 20  ataiymgq  0.999470      1454
 21  yyjmdisx  0.281790       997
 22  ebovjzjm  0.624494      3347
 23  hkdfojks  0.186570      4809
 24  pinankgn  0.171338      3443
 25  trlmmgqc  0.255069       580
 26  uzzmqzqi  0.125799       858
 27  rcvsmagz  0.584263      5189
 28  klvkvtik 

In [4]:
all_df:list[pd.DataFrame] = [df_part_1, df_part_2, df_part_3]
for idx, part in enumerate(all_df):
    part.to_parquet(f'../../data/test/dl/part_{idx}.parquet')

## Push data to MinIO

In [5]:
client = Minio(
    endpoint='localhost:9099',
    access_key='minio_access_key',
    secret_key='minio_secret_key',
    secure=False,
)

In [6]:
bucket_name = 'store'
found = client.bucket_exists(bucket_name=bucket_name)
if not found:
    client.make_bucket(bucket_name=bucket_name)
else:
    print(f'Bucket {bucket_name} already exists, skip creating!')

Bucket store already exists, skip creating!


In [8]:
for fn in os.listdir('../../data/test/dl'):
    fp = os.path.join('../../data/test/dl', fn)
    print(f"Uploading {fp}")
    client.fput_object(
        bucket_name=bucket_name,
        object_name=os.path.join('product', fn),
        file_path=fp
    )

Uploading ../../data/test/dl/part_0.parquet
Uploading ../../data/test/dl/part_1.parquet
Uploading ../../data/test/dl/part_2.parquet


## Query from Trino

- Access to `Trino`:

    ```bash
    docker exec -it datalake-trino /bin/bash
    trino
    ```

- Run the following SQL queries:

    ```sql
    CREATE SCHEMA IF NOT EXISTS datalake.store
        WITH (location = 's3://store/');

    CREATE TABLE IF NOT EXISTS datalake.store.product ( -- catalog.schema.table
        "name" VARCHAR(20),
        "price" DOUBLE,
        "quantity" INT
    ) WITH (
        external_location = 's3://store/product',
        format = 'PARQUET'
    );
    ```